# Federated Learning + Differential Privacy — Kaggle Notebook
**FL-Pneumonia-Detection | Notebook 05**

This notebook runs the full FL simulation on Kaggle (free GPU).

### What it does
1. Partitions the Kaggle chest X-ray dataset across 3 simulated hospitals (IID + Non-IID)
2. Runs FedAvg FL simulation **without** DP (baseline FL)
3. Runs FedAvg FL simulation **with** Opacus DP (ε ≤ 10, δ ≤ 1e-5)
4. Evaluates both on the held-out test set after every round
5. Logs everything to WandB for the live demo dashboard
6. Saves the best checkpoint for Azure deployment

### Before running
- Add `WANDB_API_KEY` to Kaggle Secrets (Settings → Environment → Secrets)
- Enable GPU accelerator (Settings → Accelerator → GPU T4 x2)
- Dataset: `paultimothymooney/chest-xray-pneumonia` must be added to this notebook

In [ ]:
# ============================================================
# CELL 1 — Install extra packages (Kaggle already has PyTorch)
# ============================================================
!pip install opacus==1.4.0 wandb torchmetrics --quiet
print('✅ Packages ready')

In [ ]:
# ============================================================
# CELL 2 — Imports + global config
# ============================================================
import os, sys, json, random
from datetime import datetime
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import wandb
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Paths ─────────────────────────────────────────────────────
DATA_ROOT   = '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray'
WORK_DIR    = '/kaggle/working'
MODELS_DIR  = f'{WORK_DIR}/models'
RESULTS_DIR = f'{WORK_DIR}/results'
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Device ────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# ── FL / DP Configuration ─────────────────────────────────────
FL_CONFIG = {
    # Federated Learning
    'num_rounds':    10,
    'num_clients':   3,
    'local_epochs':  3,      # keep low for Kaggle GPU quota
    'learning_rate': 1e-3,
    'lr_decay':      0.95,   # lr * decay^round
    'batch_size':    32,
    # Differential Privacy
    'target_epsilon': 10.0,  # ε ≤ 10  (project requirement)
    'target_delta':   1e-5,  # δ ≤ 10^-5
    'max_grad_norm':  1.0,   # gradient clipping bound
}

print('\n✅ Config:')
print(json.dumps(FL_CONFIG, indent=2))

In [ ]:
# ============================================================
# CELL 3 — WandB login via Kaggle secret
# ============================================================
from kaggle_secrets import UserSecretsClient
wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
wandb.login(key=wandb_key)
print('✅ WandB logged in')

In [ ]:
# ============================================================
# CELL 4 — Dataset class + transforms
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class ChestXrayDataset(Dataset):
    """
    Loads chest X-ray images from a directory with NORMAL/ and PNEUMONIA/ subdirs.
    Can also be constructed from pre-built image/label lists (used by PneumoniaClient).
    Labels: 0 = NORMAL, 1 = PNEUMONIA
    """
    CLASSES = ['NORMAL', 'PNEUMONIA']

    def __init__(self, data_dir=None, transform=None, images=None, labels=None):
        self.transform = transform
        if images is not None:
            self.images = list(images)
            self.labels = list(labels)
        else:
            self.images, self.labels = [], []
            for label, cls in enumerate(self.CLASSES):
                cls_dir = os.path.join(data_dir, cls)
                if not os.path.isdir(cls_dir):
                    continue
                for fname in sorted(os.listdir(cls_dir)):
                    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(cls_dir, fname))
                        self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_test_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


# ── Verify dataset paths ──────────────────────────────────────
for split in ['train', 'val', 'test']:
    ds = ChestXrayDataset(os.path.join(DATA_ROOT, split))
    n, p = ds.labels.count(0), ds.labels.count(1)
    print(f'{split:5s}: {len(ds):5d} images  |  NORMAL={n}  PNEUMONIA={p}')

In [ ]:
# ============================================================
# CELL 5 — Data partitioning (index-based, no file copying)
# ============================================================

# Base train dataset — no transforms yet (applied per-client)
base_train = ChestXrayDataset(os.path.join(DATA_ROOT, 'train'))


def create_iid_partitions(dataset, num_hospitals, seed=42):
    """Split dataset indices equally and randomly — same distribution per hospital."""
    rng = np.random.default_rng(seed)
    indices = np.arange(len(dataset))
    rng.shuffle(indices)
    splits = np.array_split(indices, num_hospitals)
    result = []
    for i, idx in enumerate(splits):
        labs = [dataset.labels[j] for j in idx]
        result.append(idx.tolist())
        print(f'  Hospital {i}: {len(idx):4d} samples | '
              f'NORMAL={labs.count(0)}  PNEUMONIA={labs.count(1)}')
    return result


def create_non_iid_partitions(dataset, num_hospitals, seed=42):
    """
    Non-IID: each hospital has a different pneumonia prevalence.
      Hospital 0 (Urban Teaching)   : 70% pneumonia
      Hospital 1 (Rural Community)  : 80% pneumonia
      Hospital 2 (Pediatric Center) : 60% pneumonia
    """
    rng      = np.random.default_rng(seed)
    ratios   = [0.70, 0.80, 0.60]
    hosp_names = ['Urban Teaching', 'Rural Community', 'Pediatric Center']

    normal_idx    = [i for i, l in enumerate(dataset.labels) if l == 0]
    pneumonia_idx = [i for i, l in enumerate(dataset.labels) if l == 1]
    rng.shuffle(normal_idx)
    rng.shuffle(pneumonia_idx)

    hospital_size = len(dataset) // num_hospitals
    partitions, n_ptr, p_ptr = [], 0, 0

    for i in range(num_hospitals):
        p_count = int(hospital_size * ratios[i])
        n_count = hospital_size - p_count
        if i == num_hospitals - 1:   # last hospital gets leftovers
            n_count = len(normal_idx)    - n_ptr
            p_count = len(pneumonia_idx) - p_ptr

        h_idx = normal_idx[n_ptr:n_ptr + n_count] + pneumonia_idx[p_ptr:p_ptr + p_count]
        rng.shuffle(h_idx)
        partitions.append(h_idx)
        n_ptr += n_count
        p_ptr += p_count

        labs = [dataset.labels[j] for j in h_idx]
        print(f'  Hospital {i} ({hosp_names[i]}): {len(h_idx):4d} samples | '
              f'NORMAL={labs.count(0)}  PNEUMONIA={labs.count(1)} '
              f'(target {ratios[i]:.0%} pneu)')

    return partitions


print('IID partitions:')
iid_partitions = create_iid_partitions(base_train, FL_CONFIG['num_clients'])

print('\nNon-IID partitions:')
non_iid_partitions = create_non_iid_partitions(base_train, FL_CONFIG['num_clients'])

In [ ]:
# ============================================================
# CELL 6 — Model (EfficientNet-B0, Opacus-compatible)
# ============================================================

def create_model():
    """
    EfficientNet-B0 with 2-class head.
    ModuleValidator.fix() converts BatchNorm → GroupNorm so Opacus
    can compute per-sample gradients.
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, 2),
    )
    model = ModuleValidator.fix(model)   # BatchNorm → GroupNorm
    return model


# Smoke-test
errors = ModuleValidator.validate(create_model(), strict=False)
print('Opacus validation errors:', errors if errors else 'None ✅')
_m = create_model()
_x = torch.randn(2, 3, 224, 224)
print(f'Forward pass: {list(_x.shape)} → {list(_m(_x).shape)} ✅')
del _m, _x

In [ ]:
# ============================================================
# CELL 7 — PneumoniaClient with optional Opacus DP
# ============================================================

class PneumoniaClient:
    """
    Simulates one hospital node in the FL system.

    fit() does local training and returns updated parameters.
    When use_dp=True, Opacus wraps the optimizer to clip gradients
    and add Gaussian noise before each parameter update.

    Epsilon budget: target_epsilon is split evenly across rounds
    so that after num_rounds the total ε ≈ target_epsilon
    (basic composition).
    """

    def __init__(self, hospital_id, dataset_indices, base_dataset, device, config):
        self.hospital_id = hospital_id
        self.device      = device
        self.config      = config
        self.n_samples   = len(dataset_indices)

        train_ds = ChestXrayDataset(
            images=[base_dataset.images[i] for i in dataset_indices],
            labels=[base_dataset.labels[i] for i in dataset_indices],
            transform=get_train_transforms(),
        )
        # drop_last=True is required by Opacus
        self.train_loader = DataLoader(
            train_ds,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=2,
            pin_memory=True,
            drop_last=True,
        )
        print(f'  Hospital {hospital_id}: {self.n_samples} samples '
              f'({len(self.train_loader)} batches/epoch)')

    def _get_parameters(self, model):
        return [v.cpu().numpy() for v in model.state_dict().values()]

    def _set_parameters(self, model, parameters):
        state = OrderedDict(
            {k: torch.tensor(v)
             for k, v in zip(model.state_dict().keys(), parameters)}
        )
        model.load_state_dict(state, strict=True)

    def fit(self, global_params, round_num, use_dp=False):
        """
        Local training for one FL round.
        Returns (updated_params, n_samples, metrics_dict).
        metrics_dict keys: loss, accuracy, epsilon (None when use_dp=False)
        """
        cfg = self.config

        # Fresh local model loaded with global weights
        model = create_model().to(self.device)
        self._set_parameters(model, global_params)

        lr        = cfg['learning_rate'] * (cfg['lr_decay'] ** round_num)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        epsilon_used = None

        if use_dp:
            # Per-round epsilon budget (basic composition across rounds)
            eps_per_round = cfg['target_epsilon'] / cfg['num_rounds']
            privacy_engine = PrivacyEngine()
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                data_loader=self.train_loader,
                target_epsilon=eps_per_round,
                target_delta=cfg['target_delta'],
                epochs=cfg['local_epochs'],
                max_grad_norm=cfg['max_grad_norm'],
            )
        else:
            train_loader = self.train_loader

        # ── Local training loop ──────────────────────────────
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for _ in range(cfg['local_epochs']):
            for images, labels in train_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                outputs = model(images)
                loss    = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * images.size(0)
                preds      = outputs.argmax(dim=1)
                correct    += (preds == labels).sum().item()
                total      += labels.size(0)

        if use_dp:
            epsilon_used = privacy_engine.get_epsilon(delta=cfg['target_delta'])

        metrics = {
            'loss':     total_loss / max(total, 1),
            'accuracy': correct    / max(total, 1),
            'epsilon':  epsilon_used,
        }
        return self._get_parameters(model), self.n_samples, metrics


print('✅ PneumoniaClient defined')

In [ ]:
# ============================================================
# CELL 8 — FedAvg aggregation + test set evaluator
# ============================================================

def federated_avg(params_list, sizes):
    """Weighted average of client parameters (FedAvg)."""
    total = sum(sizes)
    return [
        sum(p * (n / total) for p, n in zip(layer_vals, sizes))
        for layer_vals in zip(*params_list)
    ]


def evaluate_global_model(global_params, test_loader, device):
    """Evaluate the global model on the held-out test set."""
    model = create_model().to(device)
    state = OrderedDict(
        {k: torch.tensor(v)
         for k, v in zip(model.state_dict().keys(), global_params)}
    )
    model.load_state_dict(state, strict=True)
    model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs     = model(images)
            total_loss += criterion(outputs, labels).item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)

    return {'test_loss': total_loss / total, 'test_accuracy': correct / total}


# Build shared test loader (used across all simulation runs)
test_ds     = ChestXrayDataset(os.path.join(DATA_ROOT, 'test'), get_test_transforms())
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
print(f'✅ Test loader ready: {len(test_ds)} images')
print('Note: Kaggle val split is only 16 images (inflated accuracy) '
      '— we evaluate on test set only.')

In [ ]:
# ============================================================
# CELL 9 — Core simulation runner
# ============================================================

def run_fl_simulation(partitions, config, use_dp, run_name, strategy_name='iid'):
    """
    Full FedAvg simulation.

    Args:
        partitions   : list of index lists (one per hospital)
        config       : FL_CONFIG dict
        use_dp       : bool — enable Opacus DP
        run_name     : WandB run name
        strategy_name: 'iid' or 'non_iid' (used in WandB metric keys)

    Returns:
        best_params  : global model weights at best test accuracy
        history      : list of per-round metric dicts
    """
    dp_label = f'DP(ε={config["target_epsilon"]})' if use_dp else 'No-DP'
    print(f'\n{"="*62}')
    print(f'  FL Simulation | {strategy_name.upper()} | {dp_label}')
    print(f'  Rounds={config["num_rounds"]} | '
          f'Clients={config["num_clients"]} | '
          f'LocalEpochs={config["local_epochs"]}')
    print(f'{"="*62}')

    # WandB run for this simulation
    wandb.finish()   # close any previous run
    wandb.init(
        project='fl-pneumonia-detection',
        name=run_name,
        config={
            **config,
            'use_dp':    use_dp,
            'strategy':  strategy_name,
        },
        reinit=True,
    )

    # Initialise clients
    print('\nInitialising hospital clients...')
    clients = [
        PneumoniaClient(
            hospital_id=cid,
            dataset_indices=partitions[cid],
            base_dataset=base_train,
            device=device,
            config=config,
        )
        for cid in range(config['num_clients'])
    ]

    # Initial global parameters
    _global_model = create_model().to(device)
    global_params = [v.cpu().numpy() for v in _global_model.state_dict().values()]
    del _global_model

    history       = []
    best_test_acc = 0.0
    best_params   = None

    for rnd in range(1, config['num_rounds'] + 1):
        print(f'\n── Round {rnd}/{config["num_rounds"]} ────────────────────────────')

        # Client training
        params_list, sizes, round_metrics = [], [], []
        for client in clients:
            updated_params, n, metrics = client.fit(global_params, rnd, use_dp=use_dp)
            params_list.append(updated_params)
            sizes.append(n)
            round_metrics.append(metrics)
            eps_str = f"{metrics['epsilon']:.4f}" if metrics['epsilon'] else 'N/A'
            print(f'  H{client.hospital_id}: '
                  f'loss={metrics["loss"]:.4f}  '
                  f'acc={metrics["accuracy"]:.4f}  '
                  f'ε={eps_str}')

        # FedAvg aggregation
        global_params = federated_avg(params_list, sizes)

        # Test set evaluation
        test_metrics = evaluate_global_model(global_params, test_loader, device)

        avg_loss = float(np.mean([m['loss']     for m in round_metrics]))
        avg_acc  = float(np.mean([m['accuracy'] for m in round_metrics]))
        eps_vals = [m['epsilon'] for m in round_metrics if m['epsilon'] is not None]
        max_eps  = float(max(eps_vals)) if eps_vals else None

        print(f'  → Avg train: loss={avg_loss:.4f}  acc={avg_acc:.4f}')
        print(f'  → Test:      acc={test_metrics["test_accuracy"]:.4f}  '
              f'loss={test_metrics["test_loss"]:.4f}')
        if max_eps:
            print(f'  → ε this round: {max_eps:.4f}  '
                  f'(cumulative ≈ {max_eps * rnd:.4f})')

        # WandB logging
        log = {
            'round':          rnd,
            'train/loss':     avg_loss,
            'train/accuracy': avg_acc,
            'test/accuracy':  test_metrics['test_accuracy'],
            'test/loss':      test_metrics['test_loss'],
        }
        if max_eps is not None:
            log['privacy/epsilon_round']      = max_eps
            log['privacy/epsilon_cumulative'] = max_eps * rnd
        wandb.log(log, step=rnd)

        # Track best model
        if test_metrics['test_accuracy'] > best_test_acc:
            best_test_acc = test_metrics['test_accuracy']
            best_params   = [p.copy() for p in global_params]

        history.append({
            'round':          rnd,
            'train_loss':     avg_loss,
            'train_accuracy': avg_acc,
            'test_accuracy':  test_metrics['test_accuracy'],
            'test_loss':      test_metrics['test_loss'],
            'epsilon':        max_eps,
        })

    print(f'\n✅ Done! Best test accuracy: {best_test_acc:.4f}')
    wandb.finish()
    return best_params, history


print('✅ Simulation runner ready')

In [ ]:
# ============================================================
# CELL 10 — Run 1: FL without DP  (IID baseline)
# ============================================================
ts = datetime.now().strftime('%m%d-%H%M')

fl_params_no_dp, fl_history_no_dp = run_fl_simulation(
    partitions=iid_partitions,
    config=FL_CONFIG,
    use_dp=False,
    run_name=f'fl-no-dp-iid-{ts}',
    strategy_name='iid',
)

with open(f'{RESULTS_DIR}/fl_history_no_dp_iid.json', 'w') as f:
    json.dump(fl_history_no_dp, f, indent=2)
print('✅ FL (no DP) results saved')

In [ ]:
# ============================================================
# CELL 11 — Run 2: FL with DP  ε=10  (IID + privacy)
# ============================================================
ts = datetime.now().strftime('%m%d-%H%M')

fl_params_dp, fl_history_dp = run_fl_simulation(
    partitions=iid_partitions,
    config=FL_CONFIG,
    use_dp=True,
    run_name=f'fl-dp-eps10-iid-{ts}',
    strategy_name='iid',
)

with open(f'{RESULTS_DIR}/fl_history_dp_eps10_iid.json', 'w') as f:
    json.dump(fl_history_dp, f, indent=2)
print('✅ FL (DP ε=10) results saved')

In [ ]:
# ============================================================
# CELL 12 — Run 3: FL with DP  ε=10  (Non-IID + privacy)
# ============================================================
ts = datetime.now().strftime('%m%d-%H%M')

fl_params_dp_noniid, fl_history_dp_noniid = run_fl_simulation(
    partitions=non_iid_partitions,
    config=FL_CONFIG,
    use_dp=True,
    run_name=f'fl-dp-eps10-noniid-{ts}',
    strategy_name='non_iid',
)

with open(f'{RESULTS_DIR}/fl_history_dp_eps10_noniid.json', 'w') as f:
    json.dump(fl_history_dp_noniid, f, indent=2)
print('✅ FL (DP ε=10, Non-IID) results saved')

In [ ]:
# ============================================================
# CELL 13 — Save best model checkpoint for Azure deployment
# ============================================================

def save_checkpoint(params, history, filename):
    model = create_model()
    state = OrderedDict(
        {k: torch.tensor(v)
         for k, v in zip(model.state_dict().keys(), params)}
    )
    model.load_state_dict(state, strict=True)

    path = f'{MODELS_DIR}/{filename}'
    torch.save({
        'model_state_dict':    model.state_dict(),
        'config':              FL_CONFIG,
        'history':             history,
        'best_test_accuracy':  max(r['test_accuracy'] for r in history),
        'model_class':         'EfficientNet-B0 (Opacus GroupNorm)',
        'classes':             ['NORMAL', 'PNEUMONIA'],
    }, path)
    print(f'✅ Saved: {path}')
    print(f'   Best test accuracy: {max(r["test_accuracy"] for r in history):.4f}')
    return path


# Save FL+DP IID model (this is the one we deploy to Azure)
best_checkpoint = save_checkpoint(
    fl_params_dp,
    fl_history_dp,
    'fl_dp_eps10_iid_best.pth',
)

# Also save FL no-DP for comparison
save_checkpoint(
    fl_params_no_dp,
    fl_history_no_dp,
    'fl_no_dp_iid_best.pth',
)

# Log to WandB as artifact so it's tracked
wandb.init(
    project='fl-pneumonia-detection',
    name=f'checkpoint-upload-{datetime.now().strftime("%m%d-%H%M")}',
    reinit=True,
)
artifact = wandb.Artifact('fl-dp-pneumonia-model', type='model')
artifact.add_file(best_checkpoint)
wandb.log_artifact(artifact)
wandb.finish()
print('✅ Checkpoint logged to WandB as artifact')

In [ ]:
# ============================================================
# CELL 14 — Results comparison table
# ============================================================
print('\n' + '='*68)
print(f'{"RESULTS SUMMARY":^68}')
print('='*68)
print(f'{"Round":<7} {"FL (no DP)":>12} {"FL+DP ε=10 (IID)":>18} '
      f'{"FL+DP ε=10 (Non-IID)":>22} {"ε used":>8}')
print('-'*68)

for r_nd, r_dp, r_nid in zip(fl_history_no_dp, fl_history_dp, fl_history_dp_noniid):
    eps_str = f"{r_dp['epsilon']:.3f}" if r_dp['epsilon'] else 'N/A'
    print(f"{r_nd['round']:<7} "
          f"{r_nd['test_accuracy']:>12.4f} "
          f"{r_dp['test_accuracy']:>18.4f} "
          f"{r_nid['test_accuracy']:>22.4f} "
          f"{eps_str:>8}")

print('='*68)
best_nd  = max(r['test_accuracy'] for r in fl_history_no_dp)
best_dp  = max(r['test_accuracy'] for r in fl_history_dp)
best_nid = max(r['test_accuracy'] for r in fl_history_dp_noniid)
print(f'\nBest FL (no DP):          {best_nd:.4f}')
print(f'Best FL+DP (IID, ε=10):   {best_dp:.4f}  '
      f'(Δ = {best_dp - best_nd:+.4f} vs no-DP)')
print(f'Best FL+DP (Non-IID, ε=10):{best_nid:.4f}  '
      f'(Δ = {best_nid - best_nd:+.4f} vs no-DP)')
print('\nNote: Kaggle val set = 16 images (inflated accuracy).')
print('All reported figures are on the 624-image TEST set.')